In [1]:
import sys; sys.path.insert(0, "..")  # or absolute path to project root
from src.pipelines.silver.pipeline_race_results import SilverPipelineRaceResults
from src.pipelines.silver.pipeline_qualifying_results import SilverPipelineQualifyingResults
from src.pipelines.gold.pipeline_race_results import GoldPipelineRaceResults
from src.pipelines.bronze.pipeline import BronzePipeline
from src.storage.local_storage_backend import LocalStorageBackend
from src.models.race_results.train import train_model, train_model_2
from src.config.paths import LOCAL_DATA_DIR

storage = LocalStorageBackend(LOCAL_DATA_DIR)

# bronze_pipeline = BronzePipeline(storage)
# bronze_pipeline.extract_bronze_data()

# silver_pipeline = SilverPipelineRaceResults(storage)
# silver_pipeline.build_silver_data()

# silver_pipeline = SilverPipelineQualifyingResults(storage)
# silver_pipeline.build_silver_data()

# gold_pipeline = GoldPipelineRaceResults(storage)
# gold_pipeline.build_gold_data()

train_model()


Baseline MAE (predict position = grid): 2.64
Number of trees used: 78
Model MAE: 2.25


In [ ]:
import sys; sys.path.insert(0, "..")  # or absolute path to project root

import joblib
import pandas as pd
from src.models.position.train import load_training_data
from src.config.paths import LOCAL_MODELS_DIR

train, test, features = load_training_data()

model = joblib.load(LOCAL_MODELS_DIR / "position" / "model_v2_up_2026.pkl")

x_test = test[features]
test = test.copy()
test["predicted"] = model.predict(x_test)
test["actual"] = test["position"]
test["error"] = test["predicted"] - test["actual"]
test["abs_error"] = test["error"].abs()
test["grid_abs_error"] = (test["grid"] - test["actual"]).abs()
test["beats_grid"] = test["abs_error"] < test["grid_abs_error"]

display_cols = [
    "season", "round", "driverId", "circuitName",
    "grid", "predicted", "actual", "error", "abs_error", "beats_grid", "status"
]
print(f"Model MAE: {test['abs_error'].mean():.2f}")
print(f"Grid MAE:  {test['grid_abs_error'].mean():.2f}")
print(f"Beats grid: {test['beats_grid'].mean():.1%}")

(
    test[display_cols]
    .sort_values("abs_error", ascending=False)
    .head(40)
    .style.format({
        "predicted": "{:.1f}",
        "actual": "{:.0f}",
        "error": "{:+.1f}",
        "abs_error": "{:.1f}",
    })
)

In [ ]:
import joblib
import pandas as pd
from src.config.paths import LOCAL_DATA_DIR, LOCAL_MODELS_DIR
from src.models.position.features import (
    add_driver_last_race_position,
    add_driver_median_position_last_3_races,
    add_constructor_median_position_last_3_races,
    add_driver_circuit_median_position_last_3_races,
    add_driver_season_mediam_position,
    add_driver_positions_gained_season_median,
    add_driver_positions_gained_career_median,
)

SEASON = 2026
ROUND = 8  # next race: Austrian GP

history = pd.read_json(LOCAL_DATA_DIR / "gold/ready_to_train.json")
history = history[history["status"].isin(["Finished", "Lapped", "+1 Lap", "+2 Laps"])]

model = joblib.load(LOCAL_MODELS_DIR / "position/model_v2_up_2026.pkl")  # your saved model
features = [
    "grid", "season", "round", "circuitName", "driverId", "constructorId",
    "driver_last_race_position", "driver_median_position_last_3_races",
    "constructor_median_position_last_3_races",
    "driver_circuit_median_position_last_3_races",
    "driver_season_median_position",
    "driver_positions_gained_season_median",
    "driver_positions_gained_career_median",
]

# Example: grid from qualifying — YOU must fill this in
upcoming = pd.DataFrame([
    {"driverId": "russell",        "constructorId": "mercedes",     "grid": 1},
    {"driverId": "leclerc",        "constructorId": "ferrari",      "grid": 2},
    {"driverId": "hamilton",       "constructorId": "ferrari",      "grid": 3},
    {"driverId": "antonelli",      "constructorId": "mercedes",   "grid": 4},
    {"driverId": "max_verstappen", "constructorId": "red_bull",     "grid": 5},
    {"driverId": "norris",         "constructorId": "mclaren",      "grid": 6},
    {"driverId": "piastri",        "constructorId": "mclaren",      "grid": 7},
    {"driverId": "hadjar",         "constructorId": "red_bull",     "grid": 8},
    {"driverId": "lawson",         "constructorId": "rb",           "grid": 9},
    {"driverId": "arvid_lindblad", "constructorId": "rb",           "grid": 10},
    {"driverId": "gasly",          "constructorId": "alpine",       "grid": 11},
    {"driverId": "bortoleto",      "constructorId": "audi",         "grid": 12},
    {"driverId": "bearman",        "constructorId": "haas",         "grid": 13},
    {"driverId": "hulkenberg",     "constructorId": "audi",         "grid": 14},
    {"driverId": "ocon",           "constructorId": "haas",         "grid": 15},
    {"driverId": "colapinto",      "constructorId": "alpine",       "grid": 16},
    {"driverId": "sainz",          "constructorId": "williams",     "grid": 17},
    {"driverId": "albon",          "constructorId": "williams",     "grid": 18},
    {"driverId": "perez",          "constructorId": "cadillac",     "grid": 19},
    {"driverId": "bottas",         "constructorId": "cadillac",     "grid": 20},
    {"driverId": "alonso",         "constructorId": "aston_martin", "grid": 21},
    {"driverId": "stroll",         "constructorId": "aston_martin", "grid": 22},
])

upcoming["season"] = SEASON
upcoming["round"] = ROUND
upcoming["circuitName"] = "Red Bull Ring"
upcoming["raceName"] = "Austrian Grand Prix"
upcoming["status"] = "Upcoming"      # placeholder
upcoming["position"] = pd.NA           # unknown — that's OK
upcoming["points"] = 0.0

combined = pd.concat([history, upcoming], ignore_index=True)

# Apply feature functions individually (skip the finished-only filter in add_features)
combined = add_driver_last_race_position(combined)
combined = add_driver_median_position_last_3_races(combined)
combined = add_constructor_median_position_last_3_races(combined)
combined = add_driver_circuit_median_position_last_3_races(combined)
combined = add_driver_season_mediam_position(combined)
combined = add_driver_positions_gained_season_median(combined)
combined = add_driver_positions_gained_career_median(combined)

predict_df = combined[(combined["season"] == SEASON) & (combined["round"] == ROUND)]
predict_df.loc[predict_df["driverId"] == "antonelli", "driver_circuit_median_position_last_3_races"] = 1.0
predict_df.loc[predict_df["driverId"] == "arvid_lindblad", "driver_circuit_median_position_last_3_races"] = 10.0
predict_df = predict_df.dropna(subset=features)

predict_df = predict_df.copy()
predict_df["predicted_position"] = model.predict(predict_df[features])

predict_df.sort_values("predicted_position")[
    ["driverId", "constructorId", "grid", "predicted_position",
     "driver_median_position_last_3_races", "driver_season_median_position"]
]